## Data Cleaning - Retail Sales Dataset

**Author:** Subrata Sarker

**Purpose:** Clean and process the retail sales dataset for analysis and dashboard

---
## **Workflow Steps:**
1. Load raw dataset
2. Inspect structure & schema
3. Clean column names
4. Convert data types
5. Handle missing values
6. Clean text & category fields
7. Fix numeric fields
8. Remove duplicates
9. Validate business logic
10. Detect & cap outliers
11. Feature engineering
12. Export cleaned dataset

In [22]:
import pandas as pd
import numpy as np
import os

RAW_DATA_PATH = "../data/raw/retail_sales_50krows.csv"
CLEAN_DATA_PATH = "../data/cleaned/retail_sales_50krows_cleaned.csv"

## 1. Load Raw Data
Let's load the initial dataset and inspect  basic structure

In [23]:
df = pd.read_csv(RAW_DATA_PATH)
df.head()

,Date,Store,Region,Product,Category,Unit_Price,Quantity,Total_Sales
0,2023-04-13,Store A,South,Tablet,Electronics,300,2,600
1,2024-03-11,Store D,East,Laptop,Electronics,900,10,9000
2,2023-09-28,Store D,West,Headphones,Accessories,75,11,825
3,2023-04-17,Store E,West,Keyboard,Accessories,40,5,200
4,2023-03-13,Store B,West,Tablet,Electronics,300,4,1200


## 2. Inspect Shape, Dtypes, Missing Values, Summary Stats
We need to understand what we are working with.

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         50000 non-null  object
 1   Store        50000 non-null  object
 2   Region       50000 non-null  object
 3   Product      50000 non-null  object
 4   Category     50000 non-null  object
 5   Unit_Price   50000 non-null  int64 
 6   Quantity     50000 non-null  int64 
 7   Total_Sales  50000 non-null  int64 
dtypes: int64(3), object(5)
memory usage: 3.1+ MB


In [25]:
df.describe(include="all")

,Date,Store,Region,Product,Category,Unit_Price,Quantity,Total_Sales
count,50000,50000,50000,50000,50000,50000.00000,50000.000000,50000.000000
unique,731,5,4,7,2,NaN,NaN,NaN
top,2024-10-18,Store A,North,Headphones,Electronics,NaN,NaN,NaN
freq,93,10108,12609,7288,28478,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,315.98360,7.475020,2363.092100
std,NaN,NaN,NaN,NaN,NaN,321.56048,4.035748,3026.522958
min,NaN,NaN,NaN,NaN,NaN,25.00000,1.000000,25.000000
25%,NaN,NaN,NaN,NaN,NaN,40.00000,4.000000,300.000000
50%,NaN,NaN,NaN,NaN,NaN,180.00000,7.000000,900.000000
75%,NaN,NaN,NaN,NaN,NaN,700.00000,11.000000,3300.000000


In [26]:
df.isna().sum()

Date           0
Store          0
Region         0
Product        0
Category       0
Unit_Price     0
Quantity       0
Total_Sales    0
dtype: int64

In [27]:
df.sample(5)

,Date,Store,Region,Product,Category,Unit_Price,Quantity,Total_Sales
46148,2023-03-21,Store D,West,Tablet,Electronics,300,13,3900
41534,2023-06-27,Store C,North,Keyboard,Accessories,40,9,360
35102,2023-03-30,Store A,West,Mouse,Accessories,25,6,150
47159,2023-07-04,Store B,North,Headphones,Accessories,75,3,225
28004,2023-11-15,Store A,East,Mouse,Accessories,25,5,125


## 3. Clean Column Names
Standardize naming convention using snake_case for consistency across Python, SQL, and BI tools.

In [28]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)
df.head()

,date,store,region,product,category,unit_price,quantity,total_sales
0,2023-04-13,Store A,South,Tablet,Electronics,300,2,600
1,2024-03-11,Store D,East,Laptop,Electronics,900,10,9000
2,2023-09-28,Store D,West,Headphones,Accessories,75,11,825
3,2023-04-17,Store E,West,Keyboard,Accessories,40,5,200
4,2023-03-13,Store B,West,Tablet,Electronics,300,4,1200


## 4. Convert Data Types  
Fix dates and numeric fields.

In [29]:
# Convert dates
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Clean numeric fields
numeric_cols = ["unit_price", "quantity", "total_sales"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "")
            .str.strip()
            .replace("", np.nan)
        )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.dtypes

date           datetime64[ns]
store                  object
region                 object
product                object
category               object
unit_price              int64
quantity                int64
total_sales             int64
dtype: object

## 5. Handle Missing Values  
Critical fields cannot be missing.  
Remaining non-critical fields can be imputed.

In [30]:
# Drop rows with missing critical fields
critical_cols = ["transaction_id", "date", "product", "unit_price", "quantity"]
df = df.dropna(subset=[col for col in critical_cols if col in df.columns])

# Impute non-critical
df = df.fillna({"region": "Unknown",
    "store": "Unknown"
})
df.isna().sum()

date           0
store          0
region         0
product        0
category       0
unit_price     0
quantity       0
total_sales    0
dtype: int64

## 6. Clean Text & Category Fields  
Standardize categorical values (trim spaces, fix casing).

In [31]:
text_cols = ["store", "region", "product", "category"]

for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.title()
        )
df.sample(5)

,date,store,region,product,category,unit_price,quantity,total_sales
25643,2024-07-20,Store E,East,Laptop,Electronics,900,13,11700
15773,2023-07-26,Store B,South,Tablet,Electronics,300,3,900
27034,2024-10-25,Store C,East,Laptop,Electronics,900,7,6300
22491,2024-09-14,Store B,East,Keyboard,Accessories,40,1,40
5217,2023-04-25,Store E,West,Smartphone,Electronics,700,13,9100


## 7. Fix Numerical Fields  
- Remove negative values  
- Recalculate `total_sales`  


In [32]:
# Remove negative values
numeric_cols = ["unit_price", "quantity", "total_sales"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

# Recalculate total sales
if {"unit_price", "quantity"}.issubset(df.columns):
    df["total_sales"] = df["unit_price"] * df["quantity"]

df.describe()

,date,unit_price,quantity,total_sales
count,50000,50000.00000,50000.000000,50000.000000
mean,2023-12-31 22:25:42.528000,315.98360,7.475020,2363.092100
min,2023-01-01 00:00:00,25.00000,1.000000,25.000000
25%,2023-07-02 00:00:00,40.00000,4.000000,300.000000
50%,2024-01-02 00:00:00,180.00000,7.000000,900.000000
75%,2024-07-02 00:00:00,700.00000,11.000000,3300.000000
max,2024-12-31 00:00:00,900.00000,14.000000,12600.000000
std,NaN,321.56048,4.035748,3026.522958


## 8. Remove duplicates

In [38]:
df = df.drop_duplicates()
df.shape

(49121, 8)

## 9. Validate Business Logic  
Ensure:
- Quantity > 0  
- Unit price > 0  
- Total sales > 0  


In [34]:
numeric_cols = ["unit_price", "quantity", "total_sales"]
df_cv = df.copy(deep=True) # cleaned and validated data

for col in numeric_cols:
    if col in df_cv.columns:
        df_cv = df_cv[df[col] > 0]

df_cv.shape

(49121, 8)

## 10. Cap Outliers (IQR Method)

In [35]:
numeric_cols = ["unit_price", "quantity", "total_sales"]
df_capped = df_cv.copy(deep=True)

for col in numeric_cols:
    if col in df_capped.columns:
        Q1 = df_capped[col].quantile(0.25) # 25%
        Q3 = df_capped[col].quantile(0.75) # 75%
        IQR = Q3 - Q1 # Interquartile Range (IQR)
        lb = Q1 + 1.5 * IQR # lower boundary
        ub = Q1 - 1.5 * IQR # upper boundary
        df_capped[col] = df_capped[col].clip(lb, ub)
    
df_capped.describe()


,date,unit_price,quantity,total_sales
count,49121,49121.000000,49121.000000,49121.000000
mean,2023-12-31 23:03:37.593290240,315.856762,7.474807,1771.229820
min,2023-01-01 00:00:00,25.000000,1.000000,25.000000
25%,2023-07-02 00:00:00,40.000000,4.000000,300.000000
50%,2024-01-02 00:00:00,180.000000,7.000000,900.000000
75%,2024-07-02 00:00:00,700.000000,11.000000,3300.000000
max,2024-12-31 00:00:00,900.000000,14.000000,4800.000000
std,NaN,321.472217,4.035276,1746.969224


## 11. Feature Engineering  
Add calendar fields & profit metrics.

In [ ]:
# Time features
def add_calendar_fields(dataframe):
    """Add day, month, year to the dataframe
    if the dataframe has a datetime column

    return a new dataframe with the features
    """
    df_cal = dataframe.copy(deep=True)

    if "date" in df_cal.columns:
        df_cal["year"] = df_cal["date"].dt.year
        df_cal["month"] = df_cal["date"].dt.month
        df_cal["quarter"] = df_cal["date"].dt.quarter
        df_cal["weekday"] = df_cal["date"].dt.day_name()
    
    return df_cal

# cleaned, validated and transformed
df_cvt = add_calendar_fields(df_cv)

# cleaned, validated, capped and transformed
df_cvct = add_calendar_fields(df_capped)

# Estimated profit model @ 35% margin
df_cvt["profit"] = df_cvt["total_sales"] * 0.35
df_cvct["profit"] = df_cvct["total_sales"] * 0.35


In [37]:
df_cv.info

<bound method DataFrame.info of             date    store region     product     category  unit_price  \
0     2023-04-13  Store A  South      Tablet  Electronics         300   
1     2024-03-11  Store D   East      Laptop  Electronics         900   
2     2023-09-28  Store D   West  Headphones  Accessories          75   
3     2023-04-17  Store E   West    Keyboard  Accessories          40   
4     2023-03-13  Store B   West      Tablet  Electronics         300   
...          ...      ...    ...         ...          ...         ...   
49995 2023-01-24  Store A   East      Tablet  Electronics         300   
49996 2024-04-07  Store D   West     Monitor  Electronics         180   
49997 2024-04-25  Store A  North  Smartphone  Electronics         700   
49998 2024-12-18  Store E   East      Laptop  Electronics         900   
49999 2024-10-19  Store B   East      Laptop  Electronics         900   

       quantity  total_sales  
0             2          600  
1            10         9000 

In [39]:
df_cvt.info

<bound method DataFrame.info of             date    store region     product     category  unit_price  \
0     2023-04-13  Store A  South      Tablet  Electronics         300   
1     2024-03-11  Store D   East      Laptop  Electronics         900   
2     2023-09-28  Store D   West  Headphones  Accessories          75   
3     2023-04-17  Store E   West    Keyboard  Accessories          40   
4     2023-03-13  Store B   West      Tablet  Electronics         300   
...          ...      ...    ...         ...          ...         ...   
49995 2023-01-24  Store A   East      Tablet  Electronics         300   
49996 2024-04-07  Store D   West     Monitor  Electronics         180   
49997 2024-04-25  Store A  North  Smartphone  Electronics         700   
49998 2024-12-18  Store E   East      Laptop  Electronics         900   
49999 2024-10-19  Store B   East      Laptop  Electronics         900   

       quantity  total_sales  year  month  quarter    weekday   profit  
0             2   

In [40]:
df_cvct.info

<bound method DataFrame.info of             date    store region     product     category  unit_price  \
0     2023-04-13  Store A  South      Tablet  Electronics         300   
1     2024-03-11  Store D   East      Laptop  Electronics         900   
2     2023-09-28  Store D   West  Headphones  Accessories          75   
3     2023-04-17  Store E   West    Keyboard  Accessories          40   
4     2023-03-13  Store B   West      Tablet  Electronics         300   
...          ...      ...    ...         ...          ...         ...   
49995 2023-01-24  Store A   East      Tablet  Electronics         300   
49996 2024-04-07  Store D   West     Monitor  Electronics         180   
49997 2024-04-25  Store A  North  Smartphone  Electronics         700   
49998 2024-12-18  Store E   East      Laptop  Electronics         900   
49999 2024-10-19  Store B   East      Laptop  Electronics         900   

       quantity  total_sales  year  month  quarter    weekday   profit  
0             2   